# Лабораторная работа — Метрики качества для классификации (завершённый вариант)

Notebook содержит законченный код: загрузка данных, предобработка, импутация, масштабирование, обучение KNN / DecisionTree / LogisticRegression, оценка по метрикам и визуализация ROC / PR кривых.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

pd.set_option('display.max_columns', 500)

print('pandas', pd.__version__)

pandas 2.3.3


In [1]:
data_path = './data/data_set.csv'

if not os.path.exists(data_path):
    print(f"Файл {data_path} не найден. Пожалуйста, помести CSV в папку ./data/ и перезапусти ноутбук.")
else:
    data = pd.read_csv(data_path, delimiter=';')
    print('Данные загружены. shape =', data.shape)
    display(data.head())


NameError: name 'os' is not defined

In [3]:
def preproc(df_input):
    drop_cols = ['EDUCATION', 'FACT_ADDRESS_PROVINCE', 'FAMILY_INCOME', 'GEN_INDUSTRY', 
                 'GEN_TITLE', 'JOB_DIR', 'MARITAL_STATUS', 'ORG_TP_FCAPITAL', 'REGION_NM', 
                 'REG_ADDRESS_PROVINCE', 'ORG_TP_STATE', 'POSTAL_ADDRESS_PROVINCE', 'TP_PROVINCE', 
                 'AGREEMENT_RK']

    df_temp = df_input.copy()
    cols_to_drop = [c for c in drop_cols if c in df_temp.columns]
    df_temp = df_temp.drop(cols_to_drop, axis=1)

    digit_cols = [c for c in ['LOAN_AVG_DLQ_AMT', 'LOAN_MAX_DLQ_AMT', 'CREDIT', 'FST_PAYMENT', 'PERSONAL_INCOME'] if c in df_temp.columns]
    if digit_cols:
        df_temp[digit_cols] = df_temp[digit_cols].replace(regex={',': '.'}).astype('float64')

    return df_temp

print('Функция preproc готова')


Функция preproc готова


In [ ]:
if 'data' in globals():
    data_preproc = preproc(data)
    print('После предобработки shape =', data_preproc.shape)
    display(data_preproc.head())
else:
    print('Данные не загружены — пропускаем этот шаг.')


In [ ]:
if 'data_preproc' in globals():
    if 'TARGET' not in data_preproc.columns:
        raise KeyError('В данных отсутствует столбец TARGET')
    label_col = data_preproc.columns == 'TARGET'
    X = data_preproc.loc[:, ~label_col].values
    y = data_preproc.loc[:, label_col].values.flatten()
    print('X shape =', X.shape)
    print('y distribution:\n', pd.Series(y).value_counts())
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y if len(np.unique(y))>1 else None)
    print('Split done. X_train shape =', X_train.shape, 'X_test shape =', X_test.shape)
else:
    print('Пропускаем train/test split — нет предобработанных данных')


In [ ]:
if 'X_train' in globals():
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    imp = SimpleImputer(strategy='mean')
    imp.fit(X_train)
    X_train = imp.transform(X_train)
    X_test = imp.transform(X_test)
    ss = StandardScaler()
    ss.fit(X_train)
    X_train = ss.transform(X_train)
    X_test = ss.transform(X_test)
    print('Imputation and scaling completed.')
else:
    print('Нет разделения на train/test — пропускаем иммутацию и масштабирование')


In [ ]:
if 'X_train' in globals():
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.linear_model import LogisticRegression
    knn = KNeighborsClassifier(n_neighbors=5)
    dt = DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=None, min_samples_split=2, min_samples_leaf=10)
    logreg = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)
    knn.fit(X_train, y_train)
    dt.fit(X_train, y_train)
    logreg.fit(X_train, y_train)
    print('Обучение завершено')
else:
    print('Пропущено обучение — нет данных')


In [ ]:
if 'X_test' in globals():
    y_test_knn = knn.predict(X_test)
    y_test_dt = dt.predict(X_test)
    y_test_logreg = logreg.predict(X_test)
    if hasattr(knn, 'predict_proba'):
        y_test_proba_knn = knn.predict_proba(X_test)[:, 1]
    else:
        y_test_proba_knn = np.zeros(len(X_test))
    y_test_proba_dt = dt.predict_proba(X_test)[:, 1]
    y_test_proba_logreg = logreg.predict_proba(X_test)[:, 1]
    print('Примеры предсказаний (метки):')
    print('Truth  :', y_test[:10])
    print('kNN    :', y_test_knn[:10])
    print('DT     :', y_test_dt[:10])
    print('LogReg :', y_test_logreg[:10])
else:
    print('Нет X_test — пропускаем прогнозы')


In [ ]:
if 'X_test' in globals():
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve, average_precision_score
    def quality_metrics_report(y_true, y_pred):
        tp = np.sum( (y_true == 1) & (y_pred == 1) )
        fp = np.sum( (y_true == 0) & (y_pred == 1) )
        fn = np.sum( (y_true == 1) & (y_pred == 0) )
        tn = np.sum( (y_true == 0) & (y_pred == 0) )
        accuracy = accuracy_score(y_true, y_pred)
        error_rate = 1 - accuracy
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        return [tp, fp, fn, tn, accuracy, error_rate, precision, recall, f1]
    metrics_report = pd.DataFrame(columns=['TP', 'FP', 'FN', 'TN', 'Accuracy', 'Error rate', 'Precision', 'Recall', 'F1'])
    metrics_report.loc['kNN', :] = quality_metrics_report(y_test, y_test_knn)
    metrics_report.loc['DT', :] = quality_metrics_report(y_test, y_test_dt)
    metrics_report.loc['LogReg', :] = quality_metrics_report(y_test, y_test_logreg)
    display(metrics_report)
    fpr_knn, tpr_knn, _ = roc_curve(y_test, y_test_proba_knn)
    auc_knn = auc(fpr_knn, tpr_knn)
    fpr_dt, tpr_dt, _ = roc_curve(y_test, y_test_proba_dt)
    auc_dt = auc(fpr_dt, tpr_dt)
    fpr_logreg, tpr_logreg, _ = roc_curve(y_test, y_test_proba_logreg)
    auc_logreg = auc(fpr_logreg, tpr_logreg)
    plt.figure(figsize=(9,6))
    plt.plot(fpr_knn, tpr_knn, linewidth=2, label=f'kNN (AUC={auc_knn:.3f})')
    plt.plot(fpr_dt, tpr_dt, linewidth=2, label=f'DT (AUC={auc_dt:.3f})')
    plt.plot(fpr_logreg, tpr_logreg, linewidth=2, label=f'LogReg (AUC={auc_logreg:.3f})')
    plt.plot([0,1],[0,1], linestyle='--', linewidth=1, label='Random')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('ROC curves')
    plt.legend(loc='lower right')
    plt.grid()
    plt.show()
    precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_test_proba_knn)
    ap_knn = average_precision_score(y_test, y_test_proba_knn)
    precision_dt, recall_dt, _ = precision_recall_curve(y_test, y_test_proba_dt)
    ap_dt = average_precision_score(y_test, y_test_proba_dt)
    precision_logreg, recall_logreg, _ = precision_recall_curve(y_test, y_test_proba_logreg)
    ap_logreg = average_precision_score(y_test, y_test_proba_logreg)
    plt.figure(figsize=(9,6))
    plt.plot(recall_knn, precision_knn, linewidth=2, label=f'kNN (AP={ap_knn:.3f})')
    plt.plot(recall_dt, precision_dt, linewidth=2, label=f'DT (AP={ap_dt:.3f})')
    plt.plot(recall_logreg, precision_logreg, linewidth=2, label=f'LogReg (AP={ap_logreg:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall curves')
    plt.legend(loc='best')
    plt.grid()
    plt.show()
    print('ROC AUC: kNN={:.4f}, DT={:.4f}, LogReg={:.4f}'.format(auc_knn, auc_dt, auc_logreg))
    print('Average precision (AP): kNN={:.4f}, DT={:.4f}, LogReg={:.4f}'.format(ap_knn, ap_dt, ap_logreg))
else:
    print('Нет X_test — пропускаем оценку')


## Краткие рекомендации по улучшению моделей

- Подбор гиперпараметров через GridSearchCV/RandomizedSearchCV.
- Работа с категориальными признаками (one-hot / target encoding).
- Балансировка классов (SMOTE, class_weight).
- Использование ансамблей (RandomForest, GradientBoosting).

Файл с ноутбуком создан. Если нужно — могу также запустить ноутбук и приложить выводы (если датасет загружен).